## MIDTERM Assignment 3: Performance Benchmarking and Optimization

---

### `STEP 1` Set Up the Environment

In [1]:
import time 
import multiprocessing 
from multiprocessing.dummy import Pool  # Thread-based "multiprocessing"
import cProfile

### `STEP 2` Create a Sequential Program

In [2]:
# Function that performs a compute-intensive task
def compute_sum(data):
    total = 0
    for x in data:
        total += x * x
    return total


# Generate a large dataset
data = list(range(1, 10_000_000))


# Measure execution time of the sequential version
start_time = time.time()

result = compute_sum(data)

end_time = time.time()
serial_time = end_time - start_time

print("Sequential Execution Time:", serial_time)

Sequential Execution Time: 0.7023916244506836


### `STEP 3` Create a Parallel Version

In [3]:
# Compute-intensive function
def compute_sum(data):
    total = 0
    for x in data:
        total += x * x
    return total

# Generate dataset
data = list(range(1, 10_000_000))

# Split dataset into chunks
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    return [data[i * chunk_size:(i + 1) * chunk_size] for i in range(num_chunks)]

# Parallel computation using threads
def parallel_compute(data):
    num_threads = multiprocessing.cpu_count()  # Use as many threads as cores
    chunks = chunk_data(data, num_threads)

    with Pool(num_threads) as pool:
        results = pool.map(compute_sum, chunks)

    return sum(results)

# Measure parallel execution
start_time = time.time()
parallel_result = parallel_compute(data)
end_time = time.time()
parallel_time = end_time - start_time

print("Parallel Execution Time:", parallel_time)

Parallel Execution Time: 1.0484790802001953


### `STEP 4` Calculate Speedup and Efficiency

In [4]:
# Compute speedup
speedup = serial_time / parallel_time
print("Speedup:", speedup)


# Compute efficiency
num_processors = multiprocessing.cpu_count()
efficiency = speedup / num_processors
print("Efficiency:", efficiency)

Speedup: 0.6699147724688697
Efficiency: 0.08373934655860871


### Result Table

In [5]:
from IPython.display import Markdown, display

display(Markdown(f"""
| Version     | Execution Time (seconds) |
|:----------- |:------------------------|
| Sequential  | {serial_time:.6f}       |
| Parallel    | {parallel_time:.6f}     |
"""))


| Version     | Execution Time (seconds) |
|:----------- |:------------------------|
| Sequential  | 0.702392       |
| Parallel    | 1.048479     |


### `STEP 5` Profile the Program (Identify Bottlenecks)

In [6]:
cProfile.run("compute_sum(data)")

         4 function calls in 1.525 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    1.524    1.524    1.524    1.524 1618094196.py:2(compute_sum)
        1    0.000    0.000    1.524    1.524 <string>:1(<module>)
        1    0.000    0.000    1.525    1.525 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}




### Observations
> From what I observed, the only function that takes the most time is the **compute_sum()** that takes 1.547 seconds, which is almost the entire runtime. Following that, it is the only function where optimization is needed. The performance bottlenecks in that part due to the loop inside compute_sum() that processes every number in the dataset and performs the square calculation for each value.

---

### `STEP 6` Apply One Simple Optimization
> Notes:
> * I have applied Option A: Reduce Loop Overhead
> * I changed the optimized function to:
```
def compute_sum_optimized(data):
        return sum(map(lambda x: x*x, data)) 
```
> **Purpose of the change:** Through researching, I have learned that map() runs in C internally, so it can be slightly faster than looping in Python. It avoids the extra overhead of a Python loop that executes line by line. Also, using the first given option did not really optimize the execution time and ended up being the slowest out of the three.

In [7]:
# Optimized Function
def compute_sum_optimized(data):
    return sum(map(lambda x: x*x, data))

# Run optimized version
start_time = time.time()

optimized_result = compute_sum_optimized(data)

end_time = time.time()
optimized_time = end_time - start_time

print("Optimized Execution Time:", optimized_time)

Optimized Execution Time: 1.0314550399780273


### `STEP 7` Re-Test Performance
> I have organized the three versions back to a table for convenience.

In [8]:
from IPython.display import Markdown, display

display(Markdown(f"""
| Version     | Execution Time (seconds) |
|:----------- |:------------------------|
| Sequential  | {serial_time:.6f}       |
| Parallel    | {parallel_time:.6f}     |
| Optimized   | {optimized_time:.6f}    |
"""))


| Version     | Execution Time (seconds) |
|:----------- |:------------------------|
| Sequential  | 0.702392       |
| Parallel    | 1.048479     |
| Optimized   | 1.031455    |
